In [ ]:
## Set up the input datasets

import sys
import os 

import pandas as pd

import numpy as np 

import matplotlib.pyplot as plt 

import time as time

import cv2 
from PIL import Image, ImageFilter

import random

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch import nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR


import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, deeplabv3_resnet50
import torchvision.models as models

from tqdm import tqdm

from sklearn.model_selection import train_test_split

In [ ]:
# Load the CSV file with labels
labels_df = pd.read_csv("/kaggle/input/bhf-data-science-centre-ecg-challenge/train_final.csv")

# Some images are broken/unreadable
# Use these lines to only read in the "valid" images to stop 
with open('/kaggle/input/bhf-reference-files/valid_images_list.txt', 'r') as file:
    valid_image_paths = [line.strip() for line in file]

with open('/kaggle/input/bhf-reference-files/valid_test_images_list.txt', 'r') as file:
    valid_test_images = [line.strip() for line in file]

In [ ]:
mean_vals = labels_df[['STTC', 'HYP', 'MI', 'CD', 'AF']].to_numpy().mean(axis=0) # AF is quite rare, consider adding in class weighting
pos_weight = (1 - mean_vals)/mean_vals

train_paths, test_paths = train_test_split(valid_image_paths, test_size=0.1, random_state=42)

## Defining Segmentation Part of the Model

In [ ]:
def generate_output(image: np.array, corners: np.ndarray, scale: tuple = None):
    """Generates a perspective-transformed output image."""
    corners = order_points(corners)  # Ensure correct point order

    if scale is not None:
        corners *= np.array(scale, dtype=np.float32)  # Vectorized multiplication

    destination_corners = find_dest(corners)  # Get transformation target points
    M = cv2.getPerspectiveTransform(corners, destination_corners)  # Perspective matrix

    max_width, max_height = map(int, destination_corners[2])  # Extract output size
    out = cv2.warpPerspective(image, M, (max_width, max_height), flags=cv2.INTER_LANCZOS4)

    return np.clip(out, 0, 255).astype(np.uint8)  # Ensure valid pixel values

def order_points(pts: np.ndarray) -> np.ndarray:
    """Rearranges points to: top-left, top-right, bottom-right, bottom-left."""
    pts = np.asarray(pts, dtype=np.float32)  # Ensure NumPy array
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)

    rect = np.zeros((4, 2), dtype=np.float32)
    rect[0] = pts[np.argmin(s)]  # Top-left
    rect[2] = pts[np.argmax(s)]  # Bottom-right
    rect[1] = pts[np.argmin(diff)]  # Top-right
    rect[3] = pts[np.argmax(diff)]  # Bottom-left

    return rect  # Already a NumPy array, no conversion needed

def find_dest(pts: np.ndarray) -> np.ndarray:
    """Computes destination points based on max width and height."""
    (tl, tr, br, bl) = pts  # Unpack ordered points

    # Compute max width and height
    width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))

    return np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)

def image_preprocess_transforms(mean=(0.4611, 0.4359, 0.3905), std=(0.2193, 0.2150, 0.2109)):
    common_transforms = T.Compose([T.ToTensor(), T.Normalize(mean, std),])
    return common_transforms

def get_model(model_path, device=None):
    checkpoints = torch.load(model_path, map_location=device, weights_only=True)
    model = deeplabv3_mobilenet_v3_large(num_classes=2, aux_loss=True).to(device)
    model.load_state_dict(checkpoints, strict=False)
    return model 

def deep_learning_scan(og_image: np.array = None, 
                       trained_model=None, 
                       image_size=384, 
                       BUFFER=10, 
                       preprocess_transforms=image_preprocess_transforms(), 
                       device='cpu'):

    half = image_size // 2
    imH, imW, C = og_image.shape
    image_model = cv2.resize(og_image, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    scale_x = imW / image_size
    scale_y = imH / image_size
    
    image_model = preprocess_transforms(image_model)
    image_model = torch.unsqueeze(image_model, dim=0)
    
    image_model = image_model.to('cpu')

    # # Device on CPU
    model_cpu = trained_model.to('cpu')
    model_cpu.eval()
    
    # Rest of your preprocessing remains the same, but use model_cpu
    with torch.no_grad():
        out = model_cpu(image_model)["out"]

    out = torch.argmax(out, dim=1, keepdims=True).permute(0, 2, 3, 1)[0].numpy().squeeze().astype(np.int32)
    r_H, r_W = out.shape

    out = np.pad(out * 255, pad_width=((half, half), (half, half)), mode='constant')

    # Edge Detection.
    canny = cv2.Canny(out.astype(np.uint8), 225, 255)
    canny = cv2.dilate(canny, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    contours, _ = cv2.findContours(canny, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if not contours: 
        return None

    page = max(contours, key=cv2.contourArea)

    epsilon = 0.02 * cv2.arcLength(page, True)
    corners = cv2.approxPolyDP(page, epsilon, True)
    corners = np.concatenate(corners).astype(np.float32)

    corners[:, 0] -= half
    corners[:, 1] -= half
    corners[:, 0] *= scale_x
    corners[:, 1] *= scale_y
    
    if not (np.all(corners.min(axis=0) >= (0, 0)) and np.all(corners.max(axis=0) <= (imW, imH))):

        left_pad, top_pad, right_pad, bottom_pad = 0, 0, 0, 0

        box_corners = cv2.boxPoints(cv2.minAreaRect(corners.reshape((-1, 1, 2)))).astype(np.int32)

        box_x_min, box_y_min = np.min(box_corners, axis=0)
        box_x_max, box_y_max = np.max(box_corners, axis=0)

        # Find corner point which doesn't satify the image constraint
        # and record the amount of shift required to make the box
        # corner satisfy the constraint

        left_pad, right_pad = max(-box_x_min, 0) + BUFFER, max(box_x_max - imW, 0) + BUFFER
        top_pad, bottom_pad = max(-box_y_min, 0) + BUFFER, max(box_y_max - imH, 0) + BUFFER

        # # new image with additional zeros pixels
        # # adjust original image within the new 'image_extended'

        image_extended = np.pad(og_image, ((top_pad, bottom_pad), (left_pad, right_pad), (0, 0)), mode='constant')

        # shifting 'box_corners' the required amount
        box_corners[:, 0] += left_pad
        box_corners[:, 1] += top_pad

        corners = box_corners
        og_image = image_extended

    corners = sorted(corners.tolist())
    return generate_output(og_image, corners)

In [ ]:
seg_model_path = '/kaggle/input/document-detection/pytorch/default/1/model_mbv3_iou_mix_2C049.pth'
seg_model = get_model(seg_model_path, 'cpu')

In [ ]:
class ECG_Dataset(Dataset):
    def __init__(self, image_paths, label_df, transforms=None, preprocess_fn=None, 
                 seg_model=None, device='cpu', new_width=2500, new_height=1200):
        self.image_paths = image_paths
        self.labels_df = label_df
        self.transforms = transforms
        self.preprocess_fn = preprocess_fn
        self.device = device
        self.local_model = seg_model
        self.new_width = new_width 
        self.new_height = new_height

    def __len__(self): 
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.preprocess_fn is not None:
            image = self.preprocess_fn(image, self.local_model)
            
        image = Image.fromarray(image)
        image = image.resize((self.new_width, self.new_height), Image.LANCZOS)
        
        if self.transforms is not None:
            image = self.transforms(image)
        
        # Extract label
        filename = os.path.basename(image_path)
        index = int(filename.rsplit('_', 1)[-1].split('.')[0])
        row = self.labels_df[self.labels_df['ID'] == index].iloc[:, 1:].values
        label = torch.tensor(row, dtype=torch.float32)

        return image, label

In [ ]:
class FastGaussianBlur(object):
    """Simplified Gaussian Blur using PIL's GaussianBlur filter"""
    def __init__(self, radius_range=(1, 3)):
        self.radius_min, self.radius_max = radius_range
    
    def __call__(self, img):
        radius = random.uniform(self.radius_min, self.radius_max)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))

class FastPaperFoldEffect(object):
    """Simplified paper fold effect"""
    def __init__(self, max_intensity=0.3):
        self.max_intensity = max_intensity
    
    def __call__(self, img):
        img_tensor = T.ToTensor()(img)
        
        # Create single vertical or horizontal fold
        h, w = img_tensor.shape[1:]
        # is_vertical = random.random() > 0.5
        fold_pos = random.uniform(0.2, 0.8)

        fold_pos_px = int(w * fold_pos)
        img_tensor[:, :, fold_pos_px-2:fold_pos_px+2] *= random.uniform(0.7, 0.9)
        
        return T.ToPILImage()(img_tensor)

In [ ]:
# Optimized transform pipeline
train_transform = T.Compose([
    # Resize early to reduce computation
    T.Resize((224, 224), interpolation=InterpolationMode.LANCZOS),
    
    # Basic augmentations (fast)
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomRotation(degrees=3),
    
    # Custom effects (simplified)
    # T.RandomApply([FastGaussianBlur(radius_range=(1, 3))], p=0.3), # Blurring actually is quite rare in the data
    T.RandomApply([FastPaperFoldEffect(max_intensity=0.3)], p=0.15), # Randomly adds a fold to the paper 
    
    # Final transforms
    # T.Resize((224, 224)),
    T.ToTensor(),
    T.Lambda(lambda x: x + torch.randn_like(x) * 0.01),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transform
val_transform = T.Compose([
    T.Resize((224, 224), interpolation=InterpolationMode.LANCZOS),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Create the datasets for training and validation
train_dataset = ECG_Dataset(train_paths, labels_df, 
                            transforms=train_transform, 
                           preprocess_fn=deep_learning_scan, 
                           seg_model=seg_model)

val_dataset = ECG_Dataset(test_paths, labels_df, 
                          transforms=val_transform, 
                         preprocess_fn=deep_learning_scan, 
                         seg_model=seg_model)

# Create the dataloaders 
train_dataloader = DataLoader(train_dataset, 
                             num_workers = 4,
                             shuffle = True, 
                             batch_size = 32,
                             pin_memory=True)

val_dataloader = DataLoader(val_dataset, 
                             num_workers = 4,
                             shuffle = True, 
                             batch_size = 32,
                             pin_memory=True)

In [ ]:
# work out device 
device = torch.device("mps" if torch.backends.mps.is_available() \
                      else ("cuda" if torch.cuda.is_available() 
                            else "cpu"))

print(device)

In [ ]:
# load model checkpoint 
model_path = "/kaggle/input/efficientnet-ecg-2/pytorch/default/1/model_checkpoint.pth"
checkpoint = torch.load(model_path, map_location=torch.device('cpu'))

In [ ]:
# Load the EfficientNetV2-S model with pre-trained weights from ImageNet
model = models.efficientnet_v2_s(weights='DEFAULT')

num_ftrs = model.classifier[1].in_features  # Get input features of the classifier layer
model.classifier[1] = nn.Linear(num_ftrs, 5)  # Set output size to 5 (for your 5 classes)

model.to(device)

model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
def smooth_labels(labels, smoothing=0.1):
    """Apply label smoothing to multi-label targets."""
    return labels * (1 - smoothing) + 0.5 * smoothing

In [ ]:
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(device)  

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor,
)

In [ ]:
from torch.optim.lr_scheduler import LambdaLR

In [ ]:
num_epochs = 50 
warmup_epochs = 5 
start_epoch = checkpoint['epoch']

In [ ]:
def linear_warmup_decay_lr(epoch, total_epochs, warmup_epochs, min_lr=1.0e-5):
    if epoch < warmup_epochs:
        lr = epoch / warmup_epochs  # Linearly warm-up from 0 to base LR
    else:
        lr = (total_epochs - epoch) / (total_epochs - warmup_epochs)  # Then decay

    return max(min_lr, lr)  # Ensure LR never goes below min_lr

In [ ]:
optimizer = optim.Adam(
    model.parameters(), 
    lr=1e-3,
    weight_decay=1e-4  # Start with modest weight decay
)

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [ ]:
scheduler = LambdaLR(
    optimizer, lr_lambda=lambda epoch: linear_warmup_decay_lr(epoch, num_epochs, warmup_epochs)
)

scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

In [ ]:
from sklearn.metrics import f1_score, hamming_loss  # Evaluation metrics

def calculate_metrics(outputs, labels):
    """Calculate multiple metrics for multi-label classification"""
    # Convert outputs to predictions (0 or 1)
    predictions = (torch.sigmoid(outputs) > 0.5).float()
    
    # Move tensors to CPU and convert to numpy for sklearn metrics
    predictions_np = predictions.cpu().numpy()
    labels_np = labels.cpu().numpy()
    
    # Exact match accuracy (all labels must match)
    exact_match = (predictions == labels).all(dim=1).float().mean().item()
    
    # Hamming loss (fraction of labels that are incorrectly predicted)
    hamming = hamming_loss(labels_np, predictions_np)
    
    # Sample-wise F1 score with zero_division handling
    try:
        f1 = f1_score(labels_np, predictions_np, average='samples', zero_division=0)
    except:
        f1 = 0.0  # or np.nan if you prefer to track when this happens
    
    # Per-class F1 scores with zero_division handling
    try:
        per_class_f1 = f1_score(labels_np, predictions_np, average=None, zero_division=0)
    except:
        per_class_f1 = np.zeros(labels_np.shape[1])  # or np.full(labels_np.shape[1], np.nan)
    
    return {
        'exact_match': exact_match,
        'hamming_loss': hamming,
        'f1_score': f1,
        'per_class_f1': per_class_f1
    }

In [ ]:
# Define where to save the model
model_save_path = "model_checkpoint.pth"

running_outputs = []

for epoch in range(start_epoch, num_epochs):
    
    for param_group in optimizer.param_groups:
        print(f"Epoch {epoch+1}: Learning Rate = {param_group['lr']:.3e}")
        
    # Training phase
    model.train()
    running_train_loss = 0.0
    train_metrics = {
        'exact_match': 0.0,
        'hamming_loss': 0.0,
        'f1_score': 0.0,
        'per_class_f1': np.zeros(5)  # Assuming 5 classes
    }
    
    for inputs, labels in tqdm(train_dataloader, desc=f"Epoch {epoch+1} - Training", unit="batch"):
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.squeeze(1)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        smoothed_labels = smooth_labels(labels, smoothing=0.05)
        loss = criterion(outputs, smoothed_labels)
        loss.backward()
        optimizer.step()
        
        # Calculate and accumulate metrics
        batch_metrics = calculate_metrics(outputs, labels)
        running_train_loss += loss.item()
        train_metrics['exact_match'] += batch_metrics['exact_match']
        train_metrics['hamming_loss'] += batch_metrics['hamming_loss']
        train_metrics['f1_score'] += batch_metrics['f1_score']
        train_metrics['per_class_f1'] += batch_metrics['per_class_f1']

    scheduler.step()
    
    # Calculate average metrics for training
    num_batches = len(train_dataloader)
    avg_train_loss = running_train_loss / num_batches
    train_metrics = {k: v / num_batches for k, v in train_metrics.items()}
    
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Training Metrics:")
    print(f"Loss: {avg_train_loss:.4f}")
    print(f"Exact Match Accuracy: {train_metrics['exact_match']:.4f}")
    print(f"Hamming Loss: {train_metrics['hamming_loss']:.4f}")
    print(f"F1 Score: {train_metrics['f1_score']:.4f}")
    print("Per-class F1 Scores:", ' '.join(f"{x:.4f}" for x in train_metrics['per_class_f1']))
    
    # Validation phase
    model.eval()
    running_val_loss = 0.0
    val_metrics = {
        'exact_match': 0.0,
        'hamming_loss': 0.0,
        'f1_score': 0.0,
        'per_class_f1': np.zeros(5)  # Assuming 5 classes
    }
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_dataloader, desc=f"Epoch {epoch+1} - Validation", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.squeeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Calculate and accumulate metrics
            batch_metrics = calculate_metrics(outputs, labels)
            running_val_loss += loss.item()
            val_metrics['exact_match'] += batch_metrics['exact_match']
            val_metrics['hamming_loss'] += batch_metrics['hamming_loss']
            val_metrics['f1_score'] += batch_metrics['f1_score']
            val_metrics['per_class_f1'] += batch_metrics['per_class_f1']
    
    # Calculate average metrics for validation
    num_val_batches = len(val_dataloader)
    avg_val_loss = running_val_loss / num_val_batches
    val_metrics = {k: v / num_val_batches for k, v in val_metrics.items()}
    
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Validation Metrics:")
    print(f"Loss: {avg_val_loss:.4f}")
    print(f"Exact Match Accuracy: {val_metrics['exact_match']:.4f}")
    print(f"Hamming Loss: {val_metrics['hamming_loss']:.4f}")
    print(f"F1 Score: {val_metrics['f1_score']:.4f}")
    print("Per-class F1 Scores:", ' '.join(f"{x:.4f}" for x in val_metrics['per_class_f1']))
    
    # Save checkpoint with metrics

    checkpoint = {
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "train_loss": avg_train_loss,
    "val_loss": avg_val_loss,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics
    }
    
    torch.save(checkpoint, model_save_path)
